### Reasoning (추론)

o3 및 o4-mini와 같은 추론 모델은 강화 학습을 통해 추론을 수행하도록 훈련된 LLM입니다. 추론 모델은 사용자에게 응답하기 전에 먼저 생각하여 , 사용자에게 응답하기 전에 긴 내부 사고 사슬을 생성합니다. 추론 모델은 복잡한 문제 해결, 코딩, 과학적 추론, 그리고 에이전트 워크플로우를 위한 다단계 계획 수립에 탁월합니다. 또한 , 경량 코딩 에이전트인 Codex CLI 에 가장 적합한 모델입니다.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [2]:
from openai import OpenAI
import json

client = OpenAI()

Model = "gpt-4.1-nano"

In [6]:
prompt = """
문자열로 표현된 행렬을 '[1,2],[3,4],[5,6]' 형식으로 받아서 같은 형식으로 전치행렬을 출력하는 파이썬 스크립트를 작성하세요.
"""

response = client.responses.create(
    model="o4-mini",
    reasoning={"effort": "medium"},
    input=[
        {
            "role": "user", 
            "content": prompt
        }
    ]
)

print(response.output_text)

아래와 같이 작성하시면 됩니다. `ast.literal_eval` 을 이용해서 문자열을 파이썬 리스트로 안전하게 파싱하고, `zip` 으로 전치한 뒤 다시 원래 포맷의 문자열로 조합합니다.

```python
#!/usr/bin/env python3
import ast

def transpose_matrix_str(matrix_str: str) -> str:
    # 입력이 '[1,2],[3,4],[5,6]' 형태이므로 양쪽에 대괄호를 하나씩 더 붙여서 literal_eval 로 파싱
    matrix = ast.literal_eval(f'[{matrix_str}]')  # => [[1,2],[3,4],[5,6]]
    # 전치
    transposed = list(zip(*matrix))              # => [(1,3,5),(2,4,6)]
    # 다시 '[1,3,5],[2,4,6]' 형태의 문자열로 조합
    return ','.join(
        '[' + ','.join(str(x) for x in row) + ']'
        for row in transposed
    )

if __name__ == '__main__':
    s = input().strip()
    print(transpose_matrix_str(s))
```

사용 예시:

```
$ python3 transpose.py
[1,2],[3,4],[5,6]
[1,3,5],[2,4,6]
```

— 입력 문자열이 `[a,b],[c,d],…` 형태라면 위 스크립트가 전치행렬을 동일한 포맷으로 출력해 줍니다.


In [7]:
import ast

def transpose_str_matrix(s: str) -> str:
    # 문자열 "[1,2],[3,4],[5,6]" → "[[1,2],[3,4],[5,6]]" 로 감싸서 literal_eval
    matrix = ast.literal_eval(f'[{s}]')
    # 전치
    transposed = list(map(list, zip(*matrix)))
    # "[[1,3,5],[2,4,6]]" → "[1,3,5],[2,4,6]"
    return ','.join('[' + ','.join(map(str, row)) + ']' for row in transposed)

s = "[1,2],[3,4],[5,6]"
print(transpose_str_matrix(s))

[1,3,5],[2,4,6]


### Python 프로젝트 계획 및 생성

In [8]:
prompt = """
사용자 질문을 받아 답변과 매핑된 데이터베이스에서 검색하는 Python 앱을 만들고 싶습니다.
유사한 답변이 있으면 일치하는 답변을 검색합니다.
만약 일치하지 않으면 사용자에게 답변을 입력하도록 요청하고
질문/답변 쌍을 데이터베이스에 저장합니다. 필요한 디렉터리 구조를 계획한 후
각 파일을 전체적으로 반환합니다.
코드 전체가 아닌 시작과 끝에만 해설을 제공하세요.
"""

response = client.responses.create(
    model="o4-mini",
    input=[
        {
            "role": "user",
            "content": prompt,
        }
    ]
)

print(response.output_text)

아래와 같이 프로젝트를 구성해 보세요.

시작 전 설명
- SQLite를 이용해 질문/답변을 저장합니다.
- difflib의 SequenceMatcher로 간단한 유사도 매칭을 합니다.
- 유사 질문이 없으면 사용자에게 답변 입력을 요청하고 DB에 저장합니다.

프로젝트 디렉터리 구조
qa_app/
  src/
    db.py
    similarity.py
    main.py
  data/           ← 빈 디렉터리(qa.db는 실행 시 생성)
  requirements.txt

──────────────────────────────────────────
파일: src/db.py
──────────────────────────────────────────
import sqlite3
import os

DB_PATH = os.path.join(os.path.dirname(__file__), '..', 'data', 'qa.db')

def get_connection():
    db_dir = os.path.dirname(DB_PATH)
    if not os.path.exists(db_dir):
        os.makedirs(db_dir)
    return sqlite3.connect(DB_PATH)

def init_db():
    conn = get_connection()
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS qa_pairs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            question TEXT,
            answer TEXT
        )
    ''')
    conn.commit()
    conn.close()

def get_all_pairs():
    conn = get_connection()
    c = conn.cursor()
    c.